# Copy-on-Write (CoW)

**Note**

Copy-on-Write is now the default with pandas 3.0.

Copy-on-Write was first introduced in version 1.5.0. Starting from version 2.0 most of the optimizations that become possible through CoW are implemented and supported. All possible optimizations are supported starting from pandas 2.1.

CoW will lead to more predictable behavior since it is not possible to update more than one object with one statement, e.g. indexing operations or methods won’t have side-effects. Additionally, through delaying copies as long as possible, the average performance and memory usage will improve.

### Previous behavior

pandas indexing behavior is tricky to understand. Some operations return views while other return copies. Depending on the result of the operation, mutating one object might accidentally mutate another:

```python
```


In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.DataFrame({"foo": [1, 2, 3], "bar" : [4, 5, 6]})
subset = df["foo"]
subset.iloc[0] = 100
df

,foo,bar
0,100,4
1,2,5
2,3,6


Mutating `subset`, e.g. updating its values, also updated `df`. The exact behavior was hard to predict. Copy-on-Write solves accidentally modifying more than one object, it explicitly disallows this. `df` is unchanged:

In [3]:
pd.options.mode.copy_on_write = True

In [4]:
df = pd.DataFrame({"foo": [1, 2, 3], "bar": [4, 5, 6]})
subset = df['foo']
subset.iloc[0] = 100
df

,foo,bar
0,1,4
1,2,5
2,3,6


### Chained Assignment
Chained assignment references a technique where an object is updated through two subsequent indexing operations, e.g.

In [5]:
df = pd.DataFrame({"foo": [1, 2, 3], "bar": [4, 5, 6]})
df["foo"][df["bar"] > 5] = 100
df

C:\Users\yugal\AppData\Local\Temp\ipykernel_24884\1757644183.py:2: ChainedAssignmentError: A value is trying to be set on a copy of a DataFrame or Series through chained assignment.
When using the Copy-on-Write mode, such chained assignment never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy.

Try using '.loc[row_indexer, col_indexer] = value' instead, to perform the assignment in a single step.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["foo"][df["bar"] > 5] = 100


,foo,bar
0,1,4
1,2,5
2,3,6


The column `foo` was updated where the column `bar` is greater than 5. This violated the CoW principles though, because it would have to modify the view `df["foo"]` and `df` in one step. Hence, chained assignment will consistently never work and raise a `ChainedAssignmentError` warning with CoW enabled:


In [6]:
df = pd.DataFrame({"foo": [1, 2, 3], "bar": [4, 5, 6]})

df["foo"][df["bar"] > 5] = 100

C:\Users\yugal\AppData\Local\Temp\ipykernel_24884\618262324.py:3: ChainedAssignmentError: A value is trying to be set on a copy of a DataFrame or Series through chained assignment.
When using the Copy-on-Write mode, such chained assignment never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy.

Try using '.loc[row_indexer, col_indexer] = value' instead, to perform the assignment in a single step.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["foo"][df["bar"] > 5] = 100


In [7]:
df.loc[df["bar"] > 5, "foo"] = 100

### Patterns to avoid

In [8]:
# No defensive copy will be performed if two objects share the same data while you are modifying one object inplace.
df = df = pd.DataFrame({"a": [1, 2, 3], "b": [4, 5, 6]})
df2 = df.reset_index(drop=True)
df2.iloc[0,0] = 100

This creates two objects that share data and thus the setitem operation will trigger a copy. This is not necessary if the initial object `df` isn’t needed anymore. Simply reassigning to the same variable will invalidate the reference that is held by the object.

In [9]:
df = pd.DataFrame({"a": [1, 2, 3], "b": [4, 5, 6]})
df = df.reset_index(drop=True)
df.iloc[0, 0] = 100

In [10]:
#No copy is necessary in this example. Creating multiple references keeps unnecessary references alive and thus will hurt performance with Copy-on-Write.